# Camada Gold — Tabelas Analíticas e View Cliente 360

Le os dados da **Silver** (tipados e limpos) e cria:
- **gold_vendas_mensais** — responde pergunta 1 (evolucao de vendas)
- **gold_clientes_segmento** — responde pergunta 2 (clientes de maior valor)
- **gold_produtos_desempenho** — responde pergunta 3 (produtos com melhor desempenho)
- **gold_suporte_resumo** — chamados por tipo e canal
- **vw_cliente_360** — view obrigatoria integrando cadastro + vendas + suporte

In [0]:
%sql
-- GOLD 1: Evolucao de vendas ao longo do tempo
-- Responde: "Como as vendas evoluíram ao longo do periodo disponivel?"
CREATE OR REPLACE TABLE `desafio-semana-2`.gold.gold_vendas_mensais AS
SELECT
    YEAR(data_venda) AS ano,
    MONTH(data_venda) AS mes,
    CONCAT(CAST(YEAR(data_venda) AS STRING), '-', LPAD(CAST(MONTH(data_venda) AS STRING), 2, '0')) AS mes_ano,
    COUNT(*) AS total_pedidos,
    SUM(quantidade) AS quantidade_vendida,
    ROUND(SUM(valor_total), 2) AS faturamento,
    ROUND(AVG(valor_total), 2) AS ticket_medio
FROM `desafio-semana-2`.silver.silver_vendas
GROUP BY YEAR(data_venda), MONTH(data_venda)
ORDER BY ano, mes;

In [0]:
%sql
-- GOLD 2: Ranking de clientes por valor
-- Responde: "Quais clientes ou segmentos apresentam maior valor para o negocio?"
CREATE OR REPLACE TABLE `desafio-semana-2`.gold.gold_clientes_segmento AS
SELECT
    c.id_cliente,
    c.nome,
    c.sexo,
    c.cidade,
    c.estado,
    COUNT(v.id_venda) AS total_pedidos,
    COALESCE(SUM(v.quantidade), 0) AS quantidade_total,
    COALESCE(ROUND(SUM(v.valor_total), 2), 0) AS faturamento_total,
    COALESCE(ROUND(AVG(v.valor_total), 2), 0) AS ticket_medio,
    RANK() OVER (ORDER BY SUM(v.valor_total) DESC) AS ranking_valor
FROM `desafio-semana-2`.silver.silver_clientes c
LEFT JOIN `desafio-semana-2`.silver.silver_vendas v ON c.id_cliente = v.id_cliente
GROUP BY c.id_cliente, c.nome, c.sexo, c.cidade, c.estado
ORDER BY ranking_valor;

In [0]:
%sql
-- GOLD 3: Desempenho por produto
-- Responde: "Quais produtos ou categorias possuem melhor desempenho comercial?"
CREATE OR REPLACE TABLE `desafio-semana-2`.gold.gold_produtos_desempenho AS
SELECT
    produto,
    COUNT(*) AS total_vendas,
    SUM(quantidade) AS quantidade_vendida,
    ROUND(SUM(valor_total), 2) AS receita,
    ROUND(AVG(valor_total), 2) AS preco_medio,
    RANK() OVER (ORDER BY SUM(valor_total) DESC) AS ranking_receita
FROM `desafio-semana-2`.silver.silver_vendas
GROUP BY produto
ORDER BY ranking_receita;

In [0]:
%sql
-- GOLD 4: Resumo de suporte por canal e tipo de problema
-- O desafio pede: "Gold suporte: Chamados por tipo, indicadores de suporte"
CREATE OR REPLACE TABLE `desafio-semana-2`.gold.gold_suporte_resumo AS
SELECT
    canal,
    tipo_problema,
    COUNT(*) AS total_chamados,
    ROUND(AVG(tempo_resolucao), 1) AS tempo_medio_resolucao,
    ROUND(AVG(satisfacao_cliente), 2) AS satisfacao_media
FROM `desafio-semana-2`.silver.silver_suporte
GROUP BY canal, tipo_problema
ORDER BY total_chamados DESC;

In [0]:
%sql
-- VIEW OBRIGATORIA: vw_cliente_360
-- Integra cadastro + vendas + suporte em uma visao unica por cliente
CREATE OR REPLACE VIEW `desafio-semana-2`.gold.vw_cliente_360 AS
SELECT
    c.id_cliente,
    c.nome,
    c.sexo,
    c.cidade,
    c.estado,
    c.data_cadastro,
    COALESCE(v.total_pedidos, 0) AS total_pedidos,
    COALESCE(v.faturamento_total, 0) AS faturamento_total,
    COALESCE(v.ticket_medio, 0) AS ticket_medio,
    COALESCE(s.total_chamados, 0) AS total_chamados,
    COALESCE(s.satisfacao_media, 0) AS satisfacao_media,
    COALESCE(s.tempo_medio_resolucao, 0) AS tempo_medio_resolucao
FROM `desafio-semana-2`.silver.silver_clientes c
LEFT JOIN (
    SELECT
        id_cliente,
        COUNT(*) AS total_pedidos,
        ROUND(SUM(valor_total), 2) AS faturamento_total,
        ROUND(AVG(valor_total), 2) AS ticket_medio
    FROM `desafio-semana-2`.silver.silver_vendas
    GROUP BY id_cliente
) v ON c.id_cliente = v.id_cliente
LEFT JOIN (
    SELECT
        id_cliente,
        COUNT(*) AS total_chamados,
        ROUND(AVG(satisfacao_cliente), 2) AS satisfacao_media,
        ROUND(AVG(tempo_resolucao), 1) AS tempo_medio_resolucao
    FROM `desafio-semana-2`.silver.silver_suporte
    GROUP BY id_cliente
) s ON c.id_cliente = s.id_cliente;

In [0]:
%sql
-- VALIDACAO: Amostras de todas as tabelas Gold
SELECT 'gold_vendas_mensais' AS tabela, * FROM `desafio-semana-2`.gold.gold_vendas_mensais LIMIT 5;
SELECT 'gold_clientes_segmento' AS tabela, * FROM `desafio-semana-2`.gold.gold_clientes_segmento LIMIT 5;
SELECT 'gold_produtos_desempenho' AS tabela, * FROM `desafio-semana-2`.gold.gold_produtos_desempenho LIMIT 5;
SELECT 'gold_suporte_resumo' AS tabela, * FROM `desafio-semana-2`.gold.gold_suporte_resumo LIMIT 5;
SELECT 'vw_cliente_360' AS tabela, * FROM `desafio-semana-2`.gold.vw_cliente_360 LIMIT 5;

In [0]:
%sql
-- METRICAS FINAIS: Resumo de todas as tabelas Gold
SELECT 'gold_vendas_mensais' AS tabela, COUNT(*) AS total_registros FROM `desafio-semana-2`.gold.gold_vendas_mensais
UNION ALL
SELECT 'gold_clientes_segmento', COUNT(*) FROM `desafio-semana-2`.gold.gold_clientes_segmento
UNION ALL
SELECT 'gold_produtos_desempenho', COUNT(*) FROM `desafio-semana-2`.gold.gold_produtos_desempenho
UNION ALL
SELECT 'gold_suporte_resumo', COUNT(*) FROM `desafio-semana-2`.gold.gold_suporte_resumo
UNION ALL
SELECT 'vw_cliente_360', COUNT(*) FROM `desafio-semana-2`.gold.vw_cliente_360;